## Step 5: Convolutional Neural Network (CNN)

In this step, a Convolutional Neural Network (CNN) is implemented to improve pneumonia detection performance. Unlike MLP models, CNNs can capture spatial patterns and hierarchical features in images, making them more suitable for medical imaging tasks.

The model processes images in their 2D form instead of flattening them, allowing it to learn meaningful structures such as textures and edges associated with pneumonia.

This step aims to significantly improve classification accuracy and demonstrate the importance of spatial feature extraction in deep learning models.


In [1]:
#  Import libraries

import os
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models

2026-05-01 04:34:39.777236: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777610079.961964      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777610080.014694      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777610080.466914      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777610080.466963      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777610080.466967      57 computation_placer.cc:177] computation placer alr

In [6]:
#  Load dataset

df = pd.read_csv("/kaggle/input/datasets/sairasagnak/balanced-data-rsna/balanced_data.csv")
print(df.shape)

(12024, 8)


In [7]:
#  Fix paths

DATA_DIR = "/kaggle/input/datasets/iamtapendu/rsna-pneumonia-processed-dataset"
IMAGE_DIR = os.path.join(DATA_DIR, "Training", "Images")

df['image_path'] = df['patientId'].apply(lambda x: os.path.join(IMAGE_DIR, x + ".png"))

df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)
print(df.shape)

(12024, 8)


In [11]:
#  Preprocessing for CNN

IMG_SIZE = 128

def preprocess_image(path):
    img = cv2.imread(path, 0)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img

In [12]:
#  Prepare dataset

df = df.sample(2000, random_state=42)

images = []
labels = []

for _, row in df.iterrows():
    img = preprocess_image(row['image_path'])
    images.append(img)
    labels.append(row['Target'])

X = np.array(images)
y = np.array(labels)

# Add channel dimension (IMPORTANT)
X = X.reshape(-1, IMG_SIZE, IMG_SIZE, 1)

print(X.shape, y.shape)

(2000, 128, 128, 1) (2000,)


In [14]:
#  Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [17]:
#  Build CNN model

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

In [18]:
#  Compile model

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [19]:
#  Train model

history = model.fit(
    X_train, y_train,
    epochs=5,
    validation_data=(X_test, y_test)
)

Epoch 1/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 380ms/step - accuracy: 0.5385 - loss: 0.8161 - val_accuracy: 0.7300 - val_loss: 0.5349
Epoch 2/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 19s 375ms/step - accuracy: 0.7012 - loss: 0.5973 - val_accuracy: 0.7400 - val_loss: 0.5109
Epoch 3/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 19s 375ms/step - accuracy: 0.7516 - loss: 0.5347 - val_accuracy: 0.7500 - val_loss: 0.5398
Epoch 4/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 403ms/step - accuracy: 0.7569 - loss: 0.5198 - val_accuracy: 0.7475 - val_loss: 0.5368
Epoch 5/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 19s 376ms/step - accuracy: 0.7729 - loss: 0.4825 - val_accuracy: 0.7300 - val_loss: 0.5941


In [20]:
# Evaluate model

loss, accuracy = model.evaluate(X_test, y_test)

print("CNN Accuracy:", accuracy)

13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 101ms/step - accuracy: 0.7366 - loss: 0.6042
CNN Accuracy: 0.7300000190734863


## Step 5: Observations

* The CNN model achieved an accuracy of approximately 73%, which is slightly lower than the baseline MLP model.
* Despite using convolutional layers, the model did not significantly improve performance due to limited training epochs and a relatively small dataset subset.
* The validation accuracy plateaued early, indicating underfitting and insufficient feature learning.
* The current CNN architecture is shallow and lacks the capacity to fully capture complex spatial patterns in chest X-ray images.
* These results highlight that simply introducing convolutional layers is not sufficient; deeper architectures, better training strategies, and larger datasets are required for meaningful performance improvement.

This step demonstrates the importance of model design and training strategy in achieving effective deep learning performance.
